# Strobe imaging of membrane charging (Figure 4), on DANDI:001273

Paper: Semenov et al. (2025), Cell Reports Physical Science 6, 102660, doi:10.1016/j.xcrp.2025.102660.
Dataset: DANDI:001273, "Kinetics of Membrane Charging and Its Dependence on Neuron Shape and Alignment
with the Electric Field", version 0.250805.1356, the most recent published version (the paper cites
none).

The claims fig4-pooled-time-constants and fig4-orientation-and-poles of
`claims/001272/10.1016-j.xcrp.2025.102660.yaml` defer their fits and readings to the claims file for
DANDI:001273, whose header items are cited as 001273-F1 to F5 and 001273-O1 to O5. That header (F2)
says the fits are run unscaled where the debris ROI is not available, recorded as a deviation. No
debris ROI is deposited, so the fits here are unscaled. This notebook first shows one experiment in
detail, then fits every experiment of 7 or 8 replicates under every reading.

| Claim | Here |
|---|---|
| fig4-pooled-time-constants | tested, unscaled (001273-F2) |
| fig4-orientation-and-poles | blocked: the cell-shape groups are not in the files (001273-O4) |

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import strobe as sb

assets = sb.list_assets()
with ThreadPoolExecutor(16) as ex:
    M = list(ex.map(sb.read_meta, assets))
meta = pd.DataFrame(M)
print("dandiset", sb.DANDISET, "version", sb.VERSION, "|", len(meta), "files")

In [ ]:
print("frames per file:", meta["shape"].astype(str).value_counts().to_dict())
print("cells (subjects):", meta.subject.nunique(), "| pulse length (us):", sorted(meta.pulse_us.unique()),
      "| nominal volts:", sorted(meta.volts.unique()))
groups = meta.groupby(["subject", "electrodes", "volts"]).agg(files=("path", "size"), backward=("backward", "sum"))
print(groups.to_string())
full = groups[groups.files >= 7]
print(f"groups of 7 or 8 replicates (experiments): {len(full)}; at 40 V nominal: {int((full.index.get_level_values('volts') == 40).sum())}; "
      f"single files: {int((groups.files == 1).sum())}; the paper reports 19 experiments")
print("electrode orientation labels:", meta.electrodes.value_counts().to_dict(),
      "| cell shape labels (elongated or random): none in the files")

## One experiment, in detail

Cell 1_20240119 with the electrodes right and left of it, 8 replicates at 40 V nominal. The
claims file's ROIs are rebuilt from the RATIO image (001273-O1 a and b), the traces are averaged over
the replicates (001273-F3) and fitted under the weighting and amplitude readings (001273-O2, O3). The
paper scaled the ROI data to a "debris" ROI to remove laser-flash fluctuations (001273-F2); no debris
ROI is deposited, so the traces are unscaled, and, as a variant made here, scaled by the whole-frame
mean of each frame. A tau within 1% of a fit bound (0.01 or 20 us) is flagged: such a fit has not
found a time constant.

In [ ]:
sel = meta[(meta.subject == "1_20240119") & (meta.electrodes == "right_left") & (meta.volts == 40)]
E = sb.load_experiment(sel.to_dict("records"))
t = E["t"]
print("replicates:", E["n_replicates"], "| cell pixels:", E["cell_px"], "| ROI pixels:",
      {f"{p}/{r}": v for (p, r), v in E["roi_px"].items()})
print("RATIO extremes in the cell:", {p: round(E["rois"][(p, "extreme")], 3) for p in ("cathodic", "anodic")})
fm = E["frame_means"]
f0 = np.nanmean(fm[:, t < 0], axis=1, keepdims=True)
whole = 100 * (fm - f0) / f0
print(f"whole-frame mean, replicate mean: SD over frames {np.nanstd(np.nanmean(whole, 0)):.2f} %, "
      f"range {np.nanmin(np.nanmean(whole, 0)):.1f} to {np.nanmax(np.nanmean(whole, 0)):.1f} %")

In [ ]:
rows = []
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
for j, scaling in enumerate(("unscaled", "whole-frame scaled (variant)")):
    for key, v in E["traces"].items():
        s = v if j == 0 else 100 * ((1 + v / 100) / (1 + whole / 100) - 1)
        y = np.nanmean(s, axis=0)
        pre_sd = np.nanstd(y[t < 0])
        late = np.nanmean(y[(t >= 1.5) & (t < 2.0)])
        ax = axes[j, 0 if key[0] == "cathodic" else 1]
        ax.plot(t, y, lw=0.8, label=f"ROI {key[1]}")
        ax.axvspan(0, 2, color="tab:blue", alpha=0.08)
        ax.set_title(f"{key[0]}, {scaling}", fontsize=9)
        for w in "ab":
            for d in "ab":
                f = sb.fit_tau(t, s, w, d)
                rows.append(dict(scaling=scaling, pole=key[0], roi=key[1], weighting=w, discharge_amp=d,
                                 pre_sd=round(pre_sd, 2), late_pulse_dF=round(late, 2),
                                 tau_charging=round(f["tau_c"], 3), tau_discharging=round(f["tau_d"], 3),
                                 c_bound=f["c_bound"], d_bound=f["d_bound"],
                                 amp=round(f["amp_c"], 2)))
for ax in axes.ravel():
    ax.set_ylim(-15, 15); ax.legend(fontsize=7)
axes[1, 0].set_xlabel("time from pulse onset (us)"); axes[1, 1].set_xlabel("time from pulse onset (us)")
plt.tight_layout(); plt.show()
fits = pd.DataFrame(rows)
print(fits.to_string(index=False))
for sc, g in fits.groupby("scaling"):
    print(f"{sc}: charging tau {g.tau_charging.min()} to {g.tau_charging.max()} us, "
          f"discharging tau {g.tau_discharging.min()} to {g.tau_discharging.max()} us over the readings; "
          f"noise before the pulse (SD) {g.pre_sd.min():.2f} to {g.pre_sd.max():.2f} %, dF late in the pulse "
          f"{g.late_pulse_dF.min():.2f} to {g.late_pulse_dF.max():.2f} %")
print("fits with a tau at a bound:", int((fits.c_bound | fits.d_bound).sum()), "of", len(fits))

In this experiment the replicate-mean traces carry a frame-to-frame fluctuation common to both
poles, which the paper removed by scaling to the debris ROI. The noise before the pulse and the dF
late in the pulse, and the spread of the fitted time constants over the readings, are printed above.

## fig4-pooled-time-constants

> Membrane charging by 2-us pulses is fast; pooled over 19 experiments, charging and discharging time
> constants averaged 0.7 and 0.91 us at the cathodic pole and 0.65 and 0.8 us at the anodic pole, so
> membranes charge fully within each sine wave cycle up to 100 to 200 kHz.

Criterion (deferred to the 001273 claims file): the mean over all experiment fits of the four time
constants, each within the paper's SEM (0.7 +/- 0.08, 0.91 +/- 0.08, 0.65 +/- 0.05, 0.8 +/- 0.07 us),
under every reading of 001273-O1 (ROI a, b), O2 (weighting a, b), O3 (discharge amplitude a, b) and
F2 (here unscaled only); blocked as methods_underspecified when the readings disagree.

Every group of 7 or 8 replicates of one cell, electrode orientation and amplitude is taken as one
experiment; the files hold 22 such groups where the paper reports 19, with no mapping, so the mean is
over all 22 (a deviation). Fits with a tau at a bound are counted as failed and left out of the mean;
their number is printed. The whole-frame scaling is shown as a variant.

In [ ]:
import time
exps = full.reset_index()
rows, snr = [], []
t_start = time.time()
for _, e in exps.iterrows():
    sel = meta[(meta.subject == e.subject) & (meta.electrodes == e.electrodes) & (meta.volts == e.volts)]
    X = sb.load_experiment(sel.to_dict("records"))
    tt = X["t"]
    fm = X["frame_means"]
    wf = 100 * (fm - np.nanmean(fm[:, tt < 0], axis=1, keepdims=True)) / np.nanmean(fm[:, tt < 0], axis=1, keepdims=True)
    name = f"{e.subject} {e.electrodes} {e.volts:g} V"
    for (pole, roi), v in X["traces"].items():
        ym = np.nanmean(v, axis=0)
        snr.append(dict(experiment=name, pole=pole, roi=roi, pre_sd=np.nanstd(ym[tt < 0]),
                        late_pulse_dF=np.nanmean(ym[(tt >= 1.5) & (tt < 2.0)])))
        for scaling in ("unscaled", "whole-frame scaled"):
            s = v if scaling == "unscaled" else 100 * ((1 + v / 100) / (1 + wf / 100) - 1)
            for w in "ab":
                for d in "ab":
                    f = sb.fit_tau(tt, s, w, d)
                    rows.append(dict(experiment=name, scaling=scaling, pole=pole, roi=roi, weighting=w,
                                     discharge_amp=d, **f))
print(f"{len(exps)} experiments loaded and fitted in {(time.time() - t_start) / 60:.1f} min")
ALL = pd.DataFrame(rows)
SNR = pd.DataFrame(snr)
print("per experiment and pole, replicate mean (ROI a / b): SD before the pulse and dF late in the pulse (%)")
print(SNR.pivot_table(index="experiment", columns=["pole", "roi"], values=["pre_sd", "late_pulse_dF"]).round(2).to_string())
print(f"|dF late in the pulse| over the SD before it: median {np.median(np.abs(SNR.late_pulse_dF) / SNR.pre_sd):.2f}, "
      f"range {np.min(np.abs(SNR.late_pulse_dF) / SNR.pre_sd):.2f} to {np.max(np.abs(SNR.late_pulse_dF) / SNR.pre_sd):.2f}")

In [ ]:
EXPECT = {("cathodic", "tau_c"): (0.7, 0.08), ("cathodic", "tau_d"): (0.91, 0.08),
          ("anodic", "tau_c"): (0.65, 0.05), ("anodic", "tau_d"): (0.8, 0.07)}
summ = []
for (scaling, roi, w, d), g in ALL.groupby(["scaling", "roi", "weighting", "discharge_amp"]):
    row = dict(scaling=scaling, reading=f"{roi}/{w}/{d}")
    inside = True
    for (pole, q), (val, tol) in EXPECT.items():
        gp = g[g.pole == pole]
        bound = gp["c_bound" if q == "tau_c" else "d_bound"]
        good = gp[gp.ok & ~bound][q]
        m, se = good.mean(), good.std(ddof=1) / np.sqrt(len(good)) if len(good) > 1 else np.nan
        lab = f"{pole[:4]} {'chg' if q == 'tau_c' else 'dis'}"
        row[lab] = f"{m:.2f} +/- {se:.2f} (n={len(good)}, bound {int(bound.sum())}, failed {int((~gp.ok).sum())})"
        inside &= bool(abs(m - val) <= tol)
        row[lab + " med"] = round(float(good.median()), 2)
    row["all_within"] = inside
    summ.append(row)
S = pd.DataFrame(summ)
print(S[[c for c in S.columns if not c.endswith(" med")]].to_string(index=False))
print("\nmedians over experiments:")
print(S[["scaling", "reading"] + [c for c in S.columns if c.endswith(" med")]].to_string(index=False))
u = S[S.scaling == "unscaled"]
print(f"\nunscaled (the claims file's reading here): all four within tolerance under {int(u.all_within.sum())} of {len(u)} readings")
print(f"whole-frame scaled (variant): {int(S[S.scaling != 'unscaled'].all_within.sum())} of {len(u)}")
ok = ALL[(ALL.scaling == "unscaled") & ALL.ok]
print(f"unscaled fits with a tau at a bound: charging {int(ok.c_bound.sum())}, discharging {int(ok.d_bound.sum())} of {len(ok)}; "
      f"fits that failed: {int((~ALL[ALL.scaling == 'unscaled'].ok).sum())}")
v = ok[~ok.c_bound]
print(f"unscaled charging tau over experiments and readings, fits not at a bound: median {v.tau_c.median():.2f} us, "
      f"IQR {v.tau_c.quantile(0.25):.2f} to {v.tau_c.quantile(0.75):.2f}, range {v.tau_c.min():.2f} to {v.tau_c.max():.2f}")

**Result: not reproduced.** Fitted unscaled, as the 001273 header (F2) prescribes when the debris ROI
is missing, no reading gives all four mean time constants within the paper's SEMs (0 of 8 readings;
0 of 8 also with the whole-frame scaling). The charging means come near the paper's under some
readings (cathodic 0.68 to 1.27 us against 0.7 +/- 0.08, anodic 0.64 to 0.77 us against 0.65 +/-
0.05), but the discharging means are higher (cathodic 1.30 to 2.35 us against 0.91, anodic 0.91 to
2.62 us against 0.8), and the SEMs over experiments, 0.13 to 0.65 us, are larger than the paper's 0.05 to 0.08 us.
The traces are noisy: the dF late in the pulse is a median 1.97 times the SD before the pulse (0.16 to
8.37). Of the 352 unscaled fits, 58 charging and 15 discharging time constants end at a fit bound and
are left out; no fit failed to converge. The mean is over the 22 experiments in the files, where the
paper reports 19, and the debris scaling the paper used is not deposited, which may account for the
difference.

## fig4-orientation-and-poles

> Aligning elongated cells with the field doubled the time constants; the maximum optical TMP and the
> time constants did not differ between the cathode- and anode-facing poles; and discharging was
> slower than charging except in elongated cells aligned with the field.

Criterion: reproduced when fig4-orientation-time-constants, fig4-no-pole-difference and
fig4-discharging-slower of the 001273 claims file are all reproduced. The orientation part needs the
cell-shape groups (001273-O4), which the files do not hold, so the claim is blocked. The two parts that
need no groups are run here on the fits above, unscaled, and reported: fig4-no-pole-difference, a
two-sided paired t test over experiments of |dFmax|, charging tau and discharging tau, cathodic
against anodic (an experiment is used for a measure when neither pole's fit is at a bound); and
fig4-discharging-slower reading (a), a two-sided paired t test of discharging against charging tau
over all experiments and both poles. The exception for elongated parallel cells needs the groups.

In [ ]:
rows = []
for (roi, w, d), g in ALL[ALL.scaling == "unscaled"].groupby(["roi", "weighting", "discharge_amp"]):
    cat = g[g.pole == "cathodic"].set_index("experiment")
    ano = g[g.pole == "anodic"].set_index("experiment")
    row = dict(reading=f"{roi}/{w}/{d}")
    any_sig = False
    for q, bcol in (("amp_c", "c_bound"), ("tau_c", "c_bound"), ("tau_d", "d_bound")):
        good = ~cat[bcol] & ~ano[bcol]
        a = cat[q][good].abs() if q == "amp_c" else cat[q][good]
        b = ano[q][good].abs() if q == "amp_c" else ano[q][good]
        tt = stats.ttest_rel(a, b)
        diff = (a - b)
        ci = stats.t.ppf(0.975, len(diff) - 1) * diff.std(ddof=1) / np.sqrt(len(diff))
        row[q] = f"{diff.mean():.2f} [{diff.mean() - ci:.2f}, {diff.mean() + ci:.2f}] p={tt.pvalue:.3f} n={len(diff)}"
        any_sig |= tt.pvalue < 0.05
    row["pole_difference"] = any_sig
    ok = g[~g.c_bound & ~g.d_bound]
    td = stats.ttest_rel(ok.tau_d, ok.tau_c)
    row["dis_minus_chg"] = f"{(ok.tau_d - ok.tau_c).mean():.2f} us, p={td.pvalue:.2g}, n={len(ok)}"
    row["dis_larger_p_0_001"] = bool((ok.tau_d - ok.tau_c).mean() > 0 and td.pvalue < 0.001)
    rows.append(row)
P = pd.DataFrame(rows)
print("cathodic minus anodic, mean [95% interval], paired t test (|dFmax| in %, tau in us):")
print(P.to_string(index=False))
print(f"\na pole difference at p < 0.05 under {int(P.pole_difference.sum())} of {len(P)} readings; "
      f"discharging larger at p < 0.001 under {int(P.dis_larger_p_0_001.sum())} of {len(P)}")

**Result: blocked.** The orientation part needs the cell-shape groups, which the files do not
hold. Of the other two parts, run on the unscaled fits: no pole difference is found in |dFmax|,
charging tau or discharging tau under any of the 8 readings (paired p from 0.25 to 0.98, n = 15 to 21),
as the claim says; discharging tau is larger than charging tau under every reading (by 0.52 to 1.56 us)
but not at p < 0.001 (p from 0.0013 to 0.22) over all experiments and both poles, reading (a) of the
001273 criterion. Its readings (b) and (c), and the exception for elongated parallel cells, need the
groups.

## Summary

| Claim | Outcome |
|---|---|
| fig4-pooled-time-constants | not reproduced (unscaled fits, 22 experiments) |
| fig4-orientation-and-poles | blocked: metadata insufficient (no cell-shape groups) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's: the ROIs are rebuilt from the RATIO image under the 001273
claims file's two readings; the fits are unscaled, as the debris ROI is not deposited; the mean is over
the 22 groups of 7 or 8 replicates in the files, where the paper reports 19 experiments; fits with a
time constant at a fit bound are left out.

What was found about the dataset: the debris ROI the paper scales to is not deposited; the
experiments do not map to the paper's 19; the cell shapes (elongated or random) are not labelled; the
first acquired frame of every file is dark.